### Tauha Imran 22i1239 - G4
#### NLP - A2 - Vectors & Language Modelling
---

### 1. OCR and Preprocessing

Extracting text from PDF judgements using:

● pdf2image → to convert PDF pages to images

● pytesseract or EasyOCR → to extract text from images


Clean and normalize the text: remove headers, footers, and line breaks; fix
hyphenation and punctuation; split into sentences.
Save each record in JSON format:
```
{
"case_id": "SCP_2025_001",
"pdf_source": "path/to/file.pdf",
"ocr_text": "Full extracted text..."
}
```

In [ ]:
# install pre req libraries
%pip install pdf2image pytesseract pillow nltk
%pip install pdf2image pytesseract easyocr nltk spacy


In [10]:
#including libraries and downloading necessary data

import os, json, re
import numpy as np
from pdf2image import convert_from_path #using this to pfd -> image conversion
import pytesseract # THE OCR TOOL OF MY CHOICE
import nltk, spacy
from nltk.tokenize import sent_tokenize, word_tokenize #tokenizer functions
nltk.download('punkt')

#PS - i installed tesseract-ocr in my system using https://github.com/UB-Mannheim/tesseract/wiki
#and added the path to system environment variables C:\Program Files\Tesseract-OCR
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
#  the line above kinda did the environment variable setting lol..

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
# Ensure required NLTK tokenizer resources are available in a local nltk_data directory
import nltk, os
nltk_data_dir = os.path.join(os.getcwd(), 'nltk_data')
os.makedirs(nltk_data_dir, exist_ok=True)
# Add our local nltk_data directory to nltk lookup paths if not already present
if nltk_data_dir not in nltk.data.path:
    nltk.data.path.append(nltk_data_dir)
# Some environments require both 'punkt' and 'punkt_tab' tokenizers (punkt_tab contains language-specific tables)
required = {'punkt':'tokenizers/punkt', 'punkt_tab':'tokenizers/punkt_tab'}
for pkg, path in required.items():
    try:
        nltk.data.find(path)
        print(f"NLTK resource '{pkg}' already present ({path})")
    except LookupError:
        print(f"Downloading NLTK resource: {pkg} to {nltk_data_dir}...")
        nltk.download(pkg, download_dir=nltk_data_dir)
print('NLTK data path:', nltk_data_dir)
print('nltk.data.path:', nltk.data.path)

[nltk_data] Downloading package punkt to
[nltk_data]     f:\Projects\NLPlayground\Assignment2_Vectors &
[nltk_data]     Language Modeling\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data]   Unzipping tokenizers\punkt.zip.


NLTK data path: f:\Projects\NLPlayground\Assignment2_Vectors & Language Modeling\nltk_data


In [12]:
from pdf2image import convert_from_path
import pytesseract
import json
import re

# function to convert pdf to text using OCR
def pdf_to_text(pdf_path):
    pages = convert_from_path(pdf_path, dpi=300) # <<---- this converts each page to an image
    text = ""
    for page in pages:
        text += pytesseract.image_to_string(page)
    return text

def clean_text(text):
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'-\s+', '', text)  # fix hyphenation
    text = re.sub(r'\s{2,}', ' ', text)
    return text.strip()

pdf_path = "data/sample_case.pdf"
raw_text = pdf_to_text(pdf_path)
cleaned_text = clean_text(raw_text)

record = {
    "case_id": "SCP_2025_001",
    "pdf_source": pdf_path,
    "ocr_text": cleaned_text
}

with open("outputs/ocr_data.json", "w", encoding="utf-8") as f:
    json.dump(record, f, indent=4)

#print some stats
print(f"Extracted {len(cleaned_text)} characters from the PDF.")

Extracted 7601 characters from the PDF.


In [13]:
#print some stats
print(f"Extracted {len(cleaned_text)} characters from the PDF.")

Extracted 7601 characters from the PDF.


---

### 2. Neural Language Model for Sentence Embeddings
Now we build a Neural Language Model (from scratch) to learn distributed vector representations (embeddings) of sentences.

1. Data Preparation
Use your provided legal text corpus.
Example:

The court found the defendant guilty.
The defendant appealed the decision.
The judge dismissed the case.

1. Tokenize text into words.
2. Build a vocabulary of unique words.
3. Create training samples depending on your chosen model:
o CBOW: Predict a word from the surrounding context.
o Skip-gram: Predict surrounding context words from a given target
word.

2. Model Implementation (Using NumPy Only)

A minimal architecture:
Input → Hidden Layer (Word Embeddings) → Output (Softmax)

Example initialisation:
vocab_size = len(vocab)
embedding_dim = 50
W1 = np.random.randn(vocab_size, embedding_dim)
W2 = np.random.randn(embedding_dim, vocab_size)
Train using cross-entropy loss and gradient descent to predict target
words.

In [15]:

# now getting the data from the .json file and loading it into memory for our nueral network model.

import json
import re
import nltk # natural language toolkit
from nltk.tokenize import sent_tokenize, word_tokenize  #functions that i'mma use for tokenization
nltk.download('punkt')  # downloading the punkt tokenizer models

#LOAD THE OCR DATA FROM JSON
with open("outputs/ocr_data.json", "r", encoding="utf-8") as f:
    record = json.load(f)

text = record["ocr_text"].lower() #making it all lowercase

#splitting it into sentences.
sentences = sent_tokenize(text)
print(f"Total sentences extracted: {len(sentences)}")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - 'C:\\Users\\LENOVO/nltk_data'
    - 'c:\\Users\\LENOVO\\AppData\\Local\\Programs\\Python\\Python313\\nltk_data'
    - 'c:\\Users\\LENOVO\\AppData\\Local\\Programs\\Python\\Python313\\share\\nltk_data'
    - 'c:\\Users\\LENOVO\\AppData\\Local\\Programs\\Python\\Python313\\lib\\nltk_data'
    - 'C:\\Users\\LENOVO\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
    - 'f:\\Projects\\NLPlayground\\Assignment2_Vectors & Language Modeling\\nltk_data'
**********************************************************************
